In [ ]:
# Install seaborn quietly (in case it is not already installed in Colab)
# Seaborn is used for nicer statistical visualizations
!pip -q install seaborn

# Import core data manipulation library
import pandas as pd

# Import numerical computation library
import numpy as np

# Import plotting library
import matplotlib.pyplot as plt

# Import seaborn for improved statistical plots
import seaborn as sns

# Import linear regression model from scikit-learn
from sklearn.linear_model import LinearRegression

# Import RMSE evaluation metric
from sklearn.metrics import mean_squared_error

In [ ]:
# Load the house prices dataset
# Make sure 'train.csv' has been uploaded to Colab or is accessible via Drive
df = pd.read_csv("train.csv")

# Display the shape of the dataset (rows, columns)
# Helps us understand dataset size
print("Dataset shape:", df.shape)

# Preview first few rows to understand structure
df.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Display descriptive statistics for SalePrice
# This shows mean, std, min, max, quartiles
df["SalePrice"].describe()

In [ ]:
# Calculate skewness of SalePrice
# Positive skew indicates right-tailed distribution
print("Skewness of SalePrice:", df["SalePrice"].skew())

In [ ]:
# Plot histogram to visually inspect distribution
# This allows us to visually inspect if the distribution is right-skewed and non-normal
plt.figure(figsize=(8,4))
sns.histplot(df["SalePrice"], kde=True)
plt.title("Distribution of SalePrice")
plt.show()

In [ ]:
# Next, we are going to transform SalesPrice to stabilize variance and reduce the aforementioned skewness by using log
# Create a new column that is the natural log of SalePrice
# np.log applies natural logarithm
df["LogSalePrice"] = np.log(df["SalePrice"])

# Check skewness after transformation
print("Skewness of LogSalePrice:", df["LogSalePrice"].skew())

# This reduced skewness of SalesPrice from 1.8828757597682129 to 0.12133506220520406

In [ ]:
# Plot distribution after log transformation
plt.figure(figsize=(8,4))
sns.histplot(df["LogSalePrice"], kde=True)
plt.title("Distribution of Log(SalePrice)")
plt.show()

In [ ]:
# Next we will look at heteroskedasticity -- does variance of errors increase as the predicted SalesPrice increase?

# Using a single predictor: GrLivArea
# Define feature matrix X (must be 2D for sklearn)
# We use double brackets to keep it as a DataFrame
X = df[["GrLivArea"]]

# Define target variable y (original SalePrice)
y = df["SalePrice"]

# Initialize linear regression model
model = LinearRegression()

# Fit the model using GrLivArea to predict SalePrice
model.fit(X, y)

# Generate predictions from the model
preds = model.predict(X)

# Calculate residuals (actual - predicted)
residuals = y - preds

In [ ]:
# Plot residuals against predicted values
# We are checking for funnel shape (heteroskedasticity)
# If residual variance increases with predicted value that indicates heteroskedasticity
plt.figure(figsize=(7,4))
plt.scatter(preds, residuals, s=10, alpha=0.6)
plt.axhline(0, color="red", linewidth=1)
plt.xlabel("Predicted Price")
plt.ylabel("Residuals")
plt.title("Residual Plot (Original Target)")
plt.show()

In [ ]:
# To reduce heteroskedasticity, we will look to log-transform the SalesPrice
# We should expect more uniform variance (less funneling)
# Define log-transformed target
y_log = df["LogSalePrice"]

# Initialize new regression model
model_log = LinearRegression()

# Fit model predicting log(price)
model_log.fit(X, y_log)

# Generate predictions on log scale
preds_log = model_log.predict(X)

# Compute residuals on log scale
residuals_log = y_log - preds_log

In [ ]:
# Plot residuals for log-transformed model
plt.figure(figsize=(7,4))
plt.scatter(preds_log, residuals_log, s=10, alpha=0.6)
plt.axhline(0, color="red", linewidth=1)
plt.xlabel("Predicted Log Price")
plt.ylabel("Residuals")
plt.title("Residual Plot (Log Target)")
plt.show()

In [ ]:
# Compare predictive performance in real-world units (dollars)
# Calculate RMSE for original model
# RMSE = root mean squared error, a statisitcal metric that measures the average difference between values predicted by a model and actual observed values
  # i.e. how far apart, on average, predictions are from true values. lower values = better fit

rmse_original = np.sqrt(mean_squared_error(y, preds))

# Convert log predictions back to dollar scale
# np.exp reverses log transformation
preds_log_back = np.exp(preds_log)

# Calculate RMSE for log model in dollar terms
rmse_log_back = np.sqrt(mean_squared_error(df["SalePrice"], preds_log_back))

print("RMSE (Original Model):", rmse_original)
print("RMSE (Log Model, Back-Transformed):", rmse_log_back)

# Since we are calculating RMSE of our log predictions model back to dollar scale, we can expect the RMSE to be larger for log than original
# The original model minimizes absolute dollar error
# Log Model minimizes percentage error
# Log model performs better on relative error vs the original model that will perform better on raw dollar error

In [ ]:
# To compare the RMSE for metric the log model is actually optimized on.
rmse_log_space = np.sqrt(mean_squared_error(y_log, preds_log))
print(rmse_log_space)

In [ ]:
# To correct back-transformation bias
sigma2 = np.var(residuals_log)
preds_log_corrected = np.exp(preds_log + sigma2 / 2)

rmse_corrected = np.sqrt(mean_squared_error(df["SalePrice"], preds_log_corrected))
print("RMSE (Bias Corrected Log Model):", rmse_corrected)

In [ ]:
# Moving on from using a single predictor (GrLivArea), let's move on to look at multi-variable regression
# We will be building two baseline models: 1) linear regression predicting SalesPrice and 2) linear regression predicting LogSalesPrice

# We will also only be using numeric variables for simplicity first, categorical encoding is a separate modeling step
# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])

# Drop the original target from features later
numeric_df.head()

In [ ]:
print(numeric_df.shape)

In [ ]:
# Let's handle missing values since linear regression cannot handle NaNs (Not a Number)
# Fill missing values with median of each column
numeric_df = numeric_df.fillna(numeric_df.median())

In [ ]:
# Defining features and targets, removing SalesPrice and LogSalesPrice from our predictors

# Define X (all numeric predictors except targets)
X = numeric_df.drop(columns=["SalePrice", "LogSalePrice"])

# Define targets
y = numeric_df["SalePrice"]
y_log = numeric_df["LogSalePrice"]

In [ ]:
# Train and test split, using the same split for both targets (original and log) to keep comparison fair.
# We should never evaluate on training data
from sklearn.model_selection import train_test_split

# Split data into 80% train, 20% test
X_train, X_test, y_train, y_test, y_log_train, y_log_test = train_test_split(
    X, y, y_log, test_size=0.2, random_state=42
)

In [ ]:
# Fit Multi-Variable Linear REgression (on Original Target -- SalesPrice)
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on test set
preds = model.predict(X_test)

# Compute RMSE
rmse_original = np.sqrt(mean_squared_error(y_test, preds))
print("Test RMSE (Original Target):", rmse_original)

In [ ]:
# Fit Multi-Variable Linear Regression (on Log Target -- logSalesPrice)
model_log = LinearRegression()
model_log.fit(X_train, y_log_train)

# Predict in log space
preds_log = model_log.predict(X_test)

# Back-transform to dollars
preds_log_back = np.exp(preds_log)

# Compute RMSE in dollar scale
rmse_log = np.sqrt(mean_squared_error(y_test, preds_log_back))
print("Test RMSE (Log Target, Back-Transformed):", rmse_log)

In [ ]:
# Additional Diagnostics
# Comparing R-squared
# $-squared from log model is in log space; RMSE comparison should be in dollar scale
print("R² (Original):", model.score(X_test, y_test))
print("R² (Log model in log space):", model_log.score(X_test, y_log_test))

In [ ]:
# Now that we have our base models we should look at regularization to handle multicollinearity adn overfitting
  # I.e. how many size-related variables are correlated (GrLivArea, TotalBsmtSF, 1stFlrSF, etc.)
  #Ridge regression will penalize large coefficients, shrink correlated predictors, reduce varianace and improve generalization

#Ridge penalizes coefficent magnitude, so we need to standardize features first

#Importing ridge + scaler so that scalign and modeling happen cleanly together
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
#Ridge on Original Target (SalesPrice)
# Create pipeline: scale → ridge
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

# Fit model
ridge_model.fit(X_train, y_train)

# Predict
ridge_preds = ridge_model.predict(X_test)

# Evaluate
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_preds))

print("Ridge RMSE (Original Target):", ridge_rmse)

In [ ]:
#Ridge on Log Target (logSalesPrice)
ridge_log_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_log_model.fit(X_train, y_log_train)

ridge_log_preds = ridge_log_model.predict(X_test)

# Back-transform
ridge_log_preds_back = np.exp(ridge_log_preds)

ridge_log_rmse = np.sqrt(mean_squared_error(y_test, ridge_log_preds_back))

print("Ridge RMSE (Log Target):", ridge_log_rmse)

In [ ]:
# Tuning Alpha (Regularization Stregnth)
# Alpha controls shrinkage strength.
# Higher alpha = more shrinkage, higher bias, lower variance

#use cross-validation to find teh best alpha
from sklearn.linear_model import RidgeCV

# Define candidate alphas
alphas = np.logspace(-3, 3, 50)

ridge_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", RidgeCV(alphas=alphas, cv=5))
])

ridge_cv.fit(X_train, y_train)

# Best alpha
best_alpha = ridge_cv.named_steps["ridge"].alpha_
print("Best alpha:", best_alpha)

# Evaluate
ridge_cv_preds = ridge_cv.predict(X_test)
ridge_cv_rmse = np.sqrt(mean_squared_error(y_test, ridge_cv_preds))

print("Ridge CV RMSE:", ridge_cv_rmse)

In [ ]:
# Next let's compare coefficient magnitudes between OLS and Ridge

# First we will Fit Scaled OLS (for fair comparison) so that both models operate in standardized feature space
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# OLS with scaling
ols_scaled = Pipeline([
    ("scaler", StandardScaler()),
    ("linear", LinearRegression())
])

ols_scaled.fit(X_train, y_train)

In [ ]:
# Extracting coefficients
# Get feature names
feature_names = X_train.columns

# Extract OLS coefficients
ols_coefs = ols_scaled.named_steps["linear"].coef_

# Extract Ridge coefficients (from previously tuned ridge_cv model)
ridge_coefs = ridge_cv.named_steps["ridge"].coef_

In [ ]:
# Creating a Comparison Table
import pandas as pd
import numpy as np

coef_comparison = pd.DataFrame({
    "Feature": feature_names,
    "OLS Coefficient": ols_coefs,
    "Ridge Coefficient": ridge_coefs,
    "Absolute OLS": np.abs(ols_coefs),
    "Absolute Ridge": np.abs(ridge_coefs)
})

# Sort by largest OLS coefficients
coef_comparison = coef_comparison.sort_values(by="Absolute OLS", ascending=False)

coef_comparison.head(15)

# We will likely see: ridge coefficients are smaller, the largest OLS coefficients shrink noticeably, whereas some unstable predictors shrink heavily. This is coefficent shirnkage.

In [ ]:
# Visualizing shrinkage of the top 15 features
import matplotlib.pyplot as plt

top_features = coef_comparison.head(15)

plt.figure(figsize=(10,6))

plt.plot(top_features["Feature"], top_features["OLS Coefficient"], marker='o', label="OLS")
plt.plot(top_features["Feature"], top_features["Ridge Coefficient"], marker='o', label="Ridge")

plt.xticks(rotation=90)
plt.legend()
plt.title("Coefficient Comparison: OLS vs Ridge")
plt.show()

In [ ]:
# Exploring Lasso
# This will shrink coefficients, force some exactly to zero and perform automatic feature selection

# Import Lasso and LassoCV
from sklearn.linear_model import Lasso, LassoCV

In [ ]:
# Fit Cross-Validated Lasso using the same standardized setup as Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Define alpha grid
alphas = np.logspace(-3, 3, 50)

lasso_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", LassoCV(alphas=alphas, cv=5, max_iter=10000))
])

lasso_cv.fit(X_train, y_train)

best_alpha_lasso = lasso_cv.named_steps["lasso"].alpha_
print("Best Lasso alpha:", best_alpha_lasso)

In [ ]:
# Evaluate Lasso
lasso_preds = lasso_cv.predict(X_test)

lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_preds))
print("Lasso RMSE:", lasso_rmse)

In [ ]:
# Extract Coefficients
lasso_coefs = lasso_cv.named_steps["lasso"].coef_

In [ ]:
# Comparing Ridge vs Lasso Coefficients
coef_compare = pd.DataFrame({
    "Feature": feature_names,
    "Ridge": ridge_cv.named_steps["ridge"].coef_,
    "Lasso": lasso_coefs
})

coef_compare["Abs_Ridge"] = np.abs(coef_compare["Ridge"])
coef_compare["Abs_Lasso"] = np.abs(coef_compare["Lasso"])

coef_compare.sort_values("Abs_Ridge", ascending=False).head(15)

In [ ]:
# Counting features selected by Lasso
num_nonzero = np.sum(lasso_coefs != 0)
print("Number of features kept by Lasso:", num_nonzero)
print("Total features:", len(lasso_coefs))

# If Lasso removes a feature, that feature's predictive power is redundant or explained by correlated variables.
# Lasso may keep one, drop the other, whereas Ridge would keep both but shrink them

In [ ]:
# src/make_plots.py
"""
Generate presentation-ready plots for the Ames Housing project.

What this script does:
- Loads processed dataset (Parquet)
- Trains RidgeCV + LassoCV using leakage-safe pipelines (median impute + scale)
- Evaluates on held-out test set
- Saves clean, stakeholder-friendly plots to reports/figures/

Run from repo root:
  python3 src/make_plots.py
"""

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error


PROCESSED_PATH = Path("data/processed/train_processed.parquet")
FIG_DIR = Path("reports/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def savefig(path: Path) -> None:
    """Save figure consistently for README/Tableau assets."""
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")


def main() -> None:
    # ----------------------------
    # 1) Load data
    # ----------------------------
    if not PROCESSED_PATH.exists():
        raise FileNotFoundError(
            f"Missing {PROCESSED_PATH}. Run: python3 src/make_dataset.py"
        )

    df = pd.read_parquet(PROCESSED_PATH)

    # ----------------------------
    # 2) Define target and features
    #    (Numeric-only baseline for clean interpretability;
    #     later you can upgrade to include categoricals.)
    # ----------------------------
    if "SalePrice" not in df.columns:
        raise ValueError("Expected target column 'SalePrice' not found.")

    y = df["SalePrice"]
    X = df.drop(columns=["SalePrice", "LogSalePrice"], errors="ignore")
    X = X.select_dtypes(include=[np.number])

    if X.shape[1] == 0:
        raise ValueError("No numeric features found after filtering.")

    feature_names = X.columns.tolist()

    # ----------------------------
    # 3) Train/test split
    # ----------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # ----------------------------
    # 4) Train RidgeCV + LassoCV
    # ----------------------------
    alphas = np.logspace(-3, 3, 50)

    ridge = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RidgeCV(alphas=alphas, cv=5)),
    ])

    lasso = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LassoCV(alphas=alphas, cv=5, max_iter=20000, random_state=42)),
    ])

    ridge.fit(X_train, y_train)
    lasso.fit(X_train, y_train)

    ridge_preds = ridge.predict(X_test)
    lasso_preds = lasso.predict(X_test)

    ridge_rmse = rmse(y_test, ridge_preds)
    lasso_rmse = rmse(y_test, lasso_preds)

    ridge_alpha = float(ridge.named_steps["model"].alpha_)
    lasso_alpha = float(lasso.named_steps["model"].alpha_)
    lasso_nonzero = int(np.sum(lasso.named_steps["model"].coef_ != 0))

    print("\nModel summary (test set):")
    print(f"RidgeCV  | alpha={ridge_alpha:.6f} | RMSE=${ridge_rmse:,.0f}")
    print(f"LassoCV  | alpha={lasso_alpha:.6f} | RMSE=${lasso_rmse:,.0f} | nonzero_coefs={lasso_nonzero}")

    # ----------------------------
    # 5) Plot A: Predicted vs Actual (Ridge)
    #    Stakeholder-friendly: "Are we close to the truth?"
    # ----------------------------
    plt.figure(figsize=(7.5, 6))
    plt.scatter(y_test, ridge_preds, s=20, alpha=0.7)
    # 45-degree reference line (perfect predictions)
    lims = [
        min(y_test.min(), ridge_preds.min()),
        max(y_test.max(), ridge_preds.max())
    ]
    plt.plot(lims, lims, linewidth=2)
    plt.xlim(lims)
    plt.ylim(lims)
    plt.title("Predicted vs Actual Sale Price (RidgeCV)")
    plt.xlabel("Actual SalePrice ($)")
    plt.ylabel("Predicted SalePrice ($)")
    plt.text(
        0.05, 0.95,
        f"Test RMSE: ${ridge_rmse:,.0f}",
        transform=plt.gca().transAxes,
        verticalalignment="top"
    )
    savefig(FIG_DIR / "predicted_vs_actual_ridge.png")

    # ----------------------------
    # 6) Plot B: Residuals vs Predicted (Ridge)
    #    Stakeholder-friendly: "Where do we systematically miss?"
    # ----------------------------
    ridge_residuals = y_test - ridge_preds

    plt.figure(figsize=(7.5, 6))
    plt.scatter(ridge_preds, ridge_residuals, s=20, alpha=0.7)
    plt.axhline(0, linewidth=2)
    plt.title("Residuals vs Predicted (RidgeCV)")
    plt.xlabel("Predicted SalePrice ($)")
    plt.ylabel("Residual (Actual - Predicted) ($)")
    savefig(FIG_DIR / "residuals_vs_predicted_ridge.png")

    # ----------------------------
    # 7) Plot C: Residual Distribution (Ridge)
    #    Stakeholder-friendly: "Typical error size + symmetry"
    # ----------------------------
    plt.figure(figsize=(7.5, 6))
    plt.hist(ridge_residuals, bins=40)
    plt.axvline(0, linewidth=2)
    plt.title("Distribution of Prediction Errors (RidgeCV)")
    plt.xlabel("Residual (Actual - Predicted) ($)")
    plt.ylabel("Count")
    savefig(FIG_DIR / "residual_distribution_ridge.png")

    # ----------------------------
    # 8) Plot D: Top Feature Effects (Coefficient Magnitudes)
    #    Note: Coefs are in standardized space (because of StandardScaler).
    #    We plot absolute value for interpretability ("which mattered most").
    # ----------------------------
    ridge_coefs = ridge.named_steps["model"].coef_
    lasso_coefs = lasso.named_steps["model"].coef_

    coef_df = pd.DataFrame({
        "Feature": feature_names,
        "RidgeCoef": ridge_coefs,
        "LassoCoef": lasso_coefs,
        "AbsRidge": np.abs(ridge_coefs),
        "AbsLasso": np.abs(lasso_coefs),
    }).sort_values("AbsRidge", ascending=False)

    top_n = 15
    top = coef_df.head(top_n).iloc[::-1]  # reverse so biggest appears at top of barh

    plt.figure(figsize=(9, 7))
    plt.barh(top["Feature"], top["AbsRidge"])
    plt.title(f"Top {top_n} Drivers by Coefficient Magnitude (RidgeCV)")
    plt.xlabel("Absolute Standardized Coefficient")
    savefig(FIG_DIR / "top_drivers_ridge_bar.png")

    # ----------------------------
    # 9) Plot E: Ridge vs Lasso Coefs (Top by Ridge magnitude)
    #    Shows shrinkage vs selection behavior
    # ----------------------------
    top20 = coef_df.head(20)
    plt.figure(figsize=(11, 6))
    plt.plot(top20["Feature"], top20["RidgeCoef"], marker="o", label="Ridge")
    plt.plot(top20["Feature"], top20["LassoCoef"], marker="o", label="Lasso")
    plt.xticks(rotation=90)
    plt.title("Ridge vs Lasso Coefficients (Top 20 by Ridge magnitude)")
    plt.xlabel("Feature")
    plt.ylabel("Standardized Coefficient")
    plt.legend()
    savefig(FIG_DIR / "ridge_vs_lasso_coefficients_top20.png")

    # ----------------------------
    # 10) (Optional but great for Tableau) Save model results for dashboarding
    #     - actual, predicted, residual, percent error
    # ----------------------------
    results = pd.DataFrame({
        "ActualSalePrice": y_test.values,
        "PredSalePrice_Ridge": ridge_preds,
        "PredSalePrice_Lasso": lasso_preds,
    })
    results["Residual_Ridge"] = results["ActualSalePrice"] - results["PredSalePrice_Ridge"]
    results["Residual_Lasso"] = results["ActualSalePrice"] - results["PredSalePrice_Lasso"]
    results["PctError_Ridge"] = results["Residual_Ridge"] / results["ActualSalePrice"]
    results["PctError_Lasso"] = results["Residual_Lasso"] / results["ActualSalePrice"]

    out_csv = Path("data/processed/model_results_testset.csv")
    results.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv}")

    print("\nDone. Add these images to README and/or Tableau:")
    print(f"- {FIG_DIR.resolve()}")


if __name__ == "__main__":
    main()